# Extracting OSM parking data

In [1]:
# Load only from python 3.12
import sys

sys.path = [p for p in sys.path if "python3.8" not in p]

In [2]:
# Uncomment for package installation
!{sys.executable} -m pip install osmnx

In [2]:
# Import
import geopandas as gpd
import pandas as pd
import osmnx as ox

In [3]:
# Extract
park = ox.features.features_from_place(
    query="Amsterdam, Netherlands",
    tags={
        "amenity": "parking",
        "parking": ["surface", "street_side", "lane", "on_kerb", "half_on_kerb", "shoulder", "layby"]
    }
)

In [6]:
# Remove point geometry
park = park[park.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index()

In [12]:
# Remove underground parking
park = park[
    park["parking"] != "underground"
]

In [20]:
# Remove unwanted columns
park = park[["id", "geometry"]]

# Reproject to RD New
park = park.to_crs(28992)

In [21]:
# Get BGT parking data
bgt_park = gpd.read_file("data/parking_space.gpkg")

In [23]:
# Combine the datasets
comb_park = gpd.GeoDataFrame(
    pd.concat([bgt_park, park], ignore_index=True),
    crs=bgt_park.crs
)

# Merge overlapping polygons
comb_park = gpd.GeoDataFrame(
    geometry=[comb_park.unary_union],
    crs=comb_park.crs
).explode(index_parts=False).reset_index(drop=True)

/tmp/ipykernel_926/789470159.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[comb_park.unary_union],


In [34]:
comb_park.to_file("data/parking_space.gpkg", layer="combined")